# Project Management

## 📖 librairies

In [1]:
# python generic
import pandas as pd
import os
import numpy as np
from pathlib import Path

## plots
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

# brightway
import bw2data as bd
import bw2calc as bc
import bw2io as bi



## ▶️ starts

In [2]:
bd.projects.set_current("review_example")

15:35:12+0200 [info     ] Applying automatic update: 4.7 database dependencies in datapackage
15:35:12+0200 [info     ] Updating all LCIA methods     


634it [02:08,  4.92it/s]


15:37:21+0200 [info     ] Updating all LCI databases    


2it [00:36, 18.16s/it]


# 📝 LCI creations

## ecoinvent management

In [4]:
username = os.getenv("EI_USERNAME")
password = os.getenv("EI_PASSWORD")
syst_mod = "cutoff"
db_version = "3.12"
db_provider = "ecoinvent"
technosphere_db_name = f"{db_provider}-{db_version}-{syst_mod}"

# Import ecoinvent if needed
if technosphere_db_name not in bd.databases:
    print(f"Importing ecoinvent {db_version} ({syst_mod})...")
    bi.import_ecoinvent_release(
        username=username,
        password=password,
        system_model=syst_mod,
        version=db_version,
    )
else:
    print(f"Database '{technosphere_db_name}' already exists.")

# Load databases
technosphere_db = bd.Database(technosphere_db_name)
biosphere_db_name = f"{db_provider}-{db_version}-biosphere"
biosphere_db = bd.Database(biosphere_db_name)

Database 'ecoinvent-3.12-cutoff' already exists.


# 🧮 LCIA methods creation

## ➕ Cumulative Exergy Demand, non-renewable, total

In [5]:
def create_combined_cexd_method():
    """Create CExD non-renewable: total method by combining energy and material non-renewable methods."""

    # Define source and target method tuples
    energy_method = ("ecoinvent-3.12", "Cumulative Exergy Demand (CExD)", "energy resources: non-renewable", "exergy content")
    material_metal_method = ("ecoinvent-3.12", "Cumulative Exergy Demand (CExD)", "material resources: metals", "exergy content")
    material_minerals_method = ("ecoinvent-3.12", "Cumulative Exergy Demand (CExD)", "material resources: minerals", "exergy content")
    target_method = ("Custom Cumulative Exergy Demand (CCExD)", "non-renewable: total", "exergy content")

    # Check if source methods exist
    missing_methods = []
    if energy_method not in bd.methods:
        missing_methods.append(str(energy_method))
    if material_metal_method not in bd.methods:
        missing_methods.append(str(material_metal_method))
    if material_minerals_method not in bd.methods:
        missing_methods.append(str(material_minerals_method))

    if missing_methods:
        print(f"❌ Error: The following source methods are not registered: {', '.join(missing_methods)}")
        print("   Please create them first using create_lcia_method()")
        return

    # Load source methods
    energy_cfs = bd.Method(energy_method).load()
    material_cfs = bd.Method(material_metal_method).load()
    material_minerals_cfs = bd.Method(material_minerals_method).load()

    # Combine CFs, summing values for duplicate flows
    combined_cfs = {}
    for flow_key, cf in energy_cfs + material_cfs + material_minerals_cfs:
        combined_cfs[flow_key] = combined_cfs.get(flow_key, 0.0) + cf

    cf_list = list(combined_cfs.items())

    # Define metadata
    method_metadata = {
        'unit': 'MJ-Eq',
        'description': 'Combined cumulative exergy demand for all non-renewable resources (energy + material)',
        'source': 'Derived from CExD energy and material non-renewable methods',
        'version': '1.0',
        'num_cfs': len(cf_list),
        'application': 'All product systems unit processes'
    }

    # Register and write method
    method_object = bd.Method(target_method)
    try:
        method_object.register(**method_metadata)
        method_object.write(cf_list)
        print(
            f"\n✅ Successfully {'overwrote' if target_method in bd.methods else 'created'} "
            f"method: {target_method}\n"
            f"   - Unit: {method_metadata['unit']}\n"
            f"   - Number of CFs: {len(cf_list)}"
        )
    except Exception as e:
        print(f"\n❌ Failed to write method: {str(e)}")
        raise

    # Verification
    if target_method in bd.methods:
        retrieved_method = bd.Method(target_method)
        loaded_cfs = retrieved_method.load()
        print(
            f"\n🔍 Method verification:\n"
            f"   Name: {retrieved_method.name}\n"
            f"   Metadata version: {retrieved_method.metadata.get('version', 'N/A')}\n"
            f"   Number of CFs loaded: {len(loaded_cfs)}"
        )
        if len(loaded_cfs) != len(cf_list):
            print(f"⚠️  Warning: CF count mismatch. Expected {len(cf_list)}, got {len(loaded_cfs)}")
        else:
            print("✅ CF count matches expected value")

        print("\nSample characterization factors (first 3):")
        for cf in loaded_cfs[:3]:
            flow = bd.get_activity(cf[0])
            print(f"   - {flow['name']}: {cf[1]} MJ-Eq")
    else:
        print(f"❌ Error: Method {target_method} not found after writing attempt")

# Call the function
create_combined_cexd_method()


✅ Successfully overwrote method: ('Custom Cumulative Exergy Demand (CCExD)', 'non-renewable: total', 'exergy content')
   - Unit: MJ-Eq
   - Number of CFs: 75

🔍 Method verification:
   Name: ('Custom Cumulative Exergy Demand (CCExD)', 'non-renewable: total', 'exergy content')
   Metadata version: 1.0
   Number of CFs loaded: 75
✅ CF count matches expected value

Sample characterization factors (first 3):
   - Coal, brown: 10.3 MJ-Eq
   - Coal, hard: 19.7 MJ-Eq
   - Energy, gross calorific value, in biomass, primary forest: 1.05 MJ-Eq


# 📊 Process path contribution plot

In [7]:
# ============================================================================
# Use bw-functional to access your database
# ============================================================================
import bw2data as bd
import bw2calc as bc

# Define the methods to evaluate
methods_to_evaluate = [
    ("Custom Cumulative Exergy Demand (CCExD)", "non-renewable: total", "exergy content"),
]

# ----------------------------------------------------------------------------
# Filter activities: reference product = "copper, cathode", exclude markets
# ----------------------------------------------------------------------------
all_activities = list(technosphere_db)  # type: ignore # Get all activities

copper_cathode_activities = []
for act in all_activities:
    ref_prod = act.get('reference product', '')
    
    # Select only activities whose reference product is exactly "copper, cathode"
    # (If your database uses slightly different wording, switch to: if 'copper, cathode' in ref_prod:)
    if ref_prod == 'copper, cathode':
        upr_name = act.get('name', '')
        # Exclude activities that have "market" in the unit process name (case-insensitive)
        if 'market' not in upr_name.lower() and 'treatment' not in upr_name.lower():
            copper_cathode_activities.append(act)

if not copper_cathode_activities:
    raise ValueError("No 'copper, cathode' production activities found (excluding markets).")

print(f"Found {len(copper_cathode_activities)} activities with reference product 'copper, cathode' (markets excluded).")

# ----------------------------------------------------------------------------
# Run LCA for each filtered activity and collect results
# ----------------------------------------------------------------------------
results = []
for act in copper_cathode_activities:
    for method_key in methods_to_evaluate:
        lca = bc.LCA({act: 1}, method_key)
        lca.lci()
        lca.lcia()
        results.append({
            'name': act.get('name', ''),
            'reference product': act.get('reference product', ''),
            'location': act.get('location', ''),
            'method': method_key,
            'score': lca.score,
        })

# Sort by decreasing score
results.sort(key=lambda r: r['score'], reverse=True)

# ----------------------------------------------------------------------------
# Print sorted results
# ----------------------------------------------------------------------------
for r in results:
    print(
        f"\n--- LCA for: {r['name']} "
        f"(ref: {r['reference product']}, location: {r['location']}) ---"
    )
    print(f"  {r['method']}: {r['score']:.6e} MJ-eq")

Found 11 activities with reference product 'copper, cathode' (markets excluded).

--- LCA for: cobalt production (ref: copper, cathode, location: GLO) ---
  ('Custom Cumulative Exergy Demand (CCExD)', 'non-renewable: total', 'exergy content'): 3.442482e+02 MJ-eq

--- LCA for: aluminium alloy production, AlLi (ref: copper, cathode, location: RoW) ---
  ('Custom Cumulative Exergy Demand (CCExD)', 'non-renewable: total', 'exergy content'): 2.754800e+02 MJ-eq

--- LCA for: platinum group metal, extraction and refinery operations (ref: copper, cathode, location: ZA) ---
  ('Custom Cumulative Exergy Demand (CCExD)', 'non-renewable: total', 'exergy content'): 2.209120e+02 MJ-eq

--- LCA for: aluminium alloy production, Metallic Matrix Composite (ref: copper, cathode, location: RoW) ---
  ('Custom Cumulative Exergy Demand (CCExD)', 'non-renewable: total', 'exergy content'): 1.974292e+02 MJ-eq

--- LCA for: aluminium alloy production, AlLi (ref: copper, cathode, location: CA-QC) ---
  ('Custom 

## 📊 plots 

### function

In [17]:
def contribution_tree_bw25(fu_activity, methods, max_depth=3, min_pct=2.5, legend_fontsize=12):
    """
    Recursive contribution tree for multiple impact methods using Brightway 25.
    
    Parameters:
    - fu_activity: Brightway activity object (the functional unit)
    - methods: List of method tuples (as they appear in bd.methods)
    - max_depth: Maximum recursion depth
    - min_pct: Minimum percentage to show (cutoff)
    - legend_fontsize: Font size for legend text (default 8)
    """
    
    # Store results for each method
    all_results = {}
    all_total_scores = {}
    
    for method_key in methods:
        method_name_str = " | ".join(method_key[-2:])  # Get readable name # Cumulative Exergy Demand and dissipated will not appear for -2
        
        print(f"\n{'#'*120}")
        print(f"# Analyzing method: {method_name_str}")
        print(f"# Method key: {method_key}")
        print(f"{'#'*120}\n")
        
        # Create LCA with the specific method
        lca = bc.LCA({fu_activity: 1}, method_key)
        lca.lci()
        lca.lcia()
        total_score = lca.score
        
        results = []
        
        def traverse(node, depth, parent_name, exchange_amount_from_parent, scaling_factor=1.0):
            """
            Recursively traverse the supply chain.
            node: Brightway activity
            depth: current depth in tree
            parent_name: name of parent activity
            exchange_amount_from_parent: amount of this activity required by parent
            scaling_factor: cumulative scaling factor from functional unit
            """
            if depth > max_depth:
                return
            
            try:
                # Calculate impact per unit of this activity
                sub_lca = bc.LCA({node: 1}, method_key)
                sub_lca.lci()
                sub_lca.lcia()
                score_per_unit = sub_lca.score
            except Exception as e:
                # Silently skip to avoid clutter
                return
            
            # Calculate impact contribution
            cumulative_amount = exchange_amount_from_parent * scaling_factor # exchanges are reference flows of ISO 14040
            impact = score_per_unit * abs(cumulative_amount)
            pct = (impact / total_score * 100) if total_score and total_score != 0 else 0
            
            # Get activity details
            activity_name = node.get('name', '?')[:65]
            ref_product = node.get('reference product', '?')[:45]
            location = node.get('location', '?')[:6]
            
            results.append({
                'depth': depth,
                'parent': parent_name,
                'activity_name': activity_name,
                'reference_product': ref_product,
                'location': location,
                'exchange_amount': exchange_amount_from_parent,
                'cumulative_amount': cumulative_amount,
                'score_per_unit': score_per_unit,
                'impact_abs': impact,
                'impact_pct': pct,
                'key': node.key
            })
            
            # Only traverse deeper if contribution exceeds threshold
            if pct >= min_pct and depth < max_depth:
                # Get technosphere exchanges (inputs) from this activity
                for exc in node.technosphere():
                    if exc.get('type') == 'technosphere':
                        supplier = exc.input
                        amount = exc.get('amount', 0)
                        
                        # Skip self-loops and zero amounts
                        if amount != 0 and supplier.key != node.key:
                            traverse(
                                supplier, 
                                depth + 1,
                                parent_name=ref_product or activity_name,
                                exchange_amount_from_parent=amount,
                                scaling_factor=cumulative_amount
                            )
        
        # Start traversal from functional unit
        fu_name = fu_activity.get('name', 'FU')
        fu_ref = fu_activity.get('reference product', fu_name)
        traverse(fu_activity, depth=1, parent_name="FU", exchange_amount_from_parent=1.0, scaling_factor=1.0)
        
        # Create DataFrame
        df = pd.DataFrame(results)
        if not df.empty:
            df = df.sort_values(['depth', 'impact_abs'], ascending=[True, False])
        else:
            df = pd.DataFrame(columns=['depth', 'parent', 'activity_name', 'reference_product', 
                                       'location', 'exchange_amount', 'cumulative_amount', 
                                       'score_per_unit', 'impact_abs', 'impact_pct', 'key'])
        
        print(f"Functional Unit: {fu_name} ({fu_activity.get('location', '?')})")
        print(f"Total {method_name_str} score: {total_score:.6e} MJ-eq\n")
        
        # Store detailed data for plotting
        plot_data = {'absolute': {}, 'relative': {}}
        
        if df.empty:
            print("No results found for this method.")
        else:
            for depth in sorted(df['depth'].unique()):
                depth_df = df[df['depth'] == depth].copy()
                
                # Filter by percentage threshold (absolute value for negative impacts)
                depth_df_filtered = depth_df[depth_df['impact_pct'].abs() >= min_pct].copy()
                
                if depth_df_filtered.empty and len(depth_df) > 0:
                    # If nothing meets threshold but there are results, take top 5
                    depth_df_filtered = depth_df.nlargest(5, 'impact_abs')
                
                if depth_df_filtered.empty:
                    continue
                
                # Calculate rest (contributions below cutoff)
                filtered_sum_abs = depth_df_filtered['impact_abs'].sum()
                filtered_sum_pct = depth_df_filtered['impact_pct'].sum()
                rest_abs = total_score - filtered_sum_abs
                rest_pct = 100.0 - filtered_sum_pct
                
                # Only add Rest if it's meaningful
                if abs(rest_pct) > 0.01 and len(depth_df) > len(depth_df_filtered):
                    rest_row = pd.DataFrame([{
                        'depth': depth,
                        'parent': '-',
                        'activity_name': f'Rest ({len(depth_df) - len(depth_df_filtered)} other processes)',
                        'reference_product': '-',
                        'location': '-',
                        'exchange_amount': 0,
                        'cumulative_amount': 0,
                        'score_per_unit': 0,
                        'impact_abs': rest_abs,
                        'impact_pct': rest_pct,
                        'key': None
                    }])
                    depth_df_filtered = pd.concat([depth_df_filtered, rest_row], ignore_index=True)
                
                # Print table
                print(f"{'='*120}")
                print(f"Depth {depth} — {len(depth_df_filtered)-1 if rest_pct else len(depth_df_filtered)} suppliers + Rest (showing >={min_pct}% of total)")
                print(f"{'='*120}")
                print(f"{'Activity':<65} {'Product':<30} {'Loc':<6} {'Impact (MJ-eq)':>15} {'%':>8}")
                print("-"*130)
                
                for _, row in depth_df_filtered.iterrows():
                    name = row['activity_name'][:63] if row['activity_name'] else '?'
                    prod = row['reference_product'][:28] if row['reference_product'] else '?'
                    loc = row['location'][:4] if row['location'] else '?'
                    print(f"{name:<65} {prod:<30} {loc:<6} {row['impact_abs']:>15.4e} {row['impact_pct']:>7.1f}%")
                
                print(f"{'':>118} {'──────':>12}")
                print(f"{'Total:':>110} {depth_df_filtered['impact_pct'].sum():>7.1f}%")
                print()
                
                # Store for plotting
                plot_data['absolute'][depth] = depth_df_filtered[['activity_name', 'impact_abs']].copy()
                plot_data['relative'][depth] = depth_df_filtered[['activity_name', 'impact_pct']].copy()
        
        all_results[method_key] = {
            'df': df,
            'total_score': total_score,
            'plot_data': plot_data,
            'method_name': method_name_str
        }
        all_total_scores[method_key] = total_score
    
    # --- Create comparison plots ---
    if not all_results:
        print("\nNo results to plot!")
        return all_results, all_total_scores
    
    # Get max depth across all methods
    max_depths = []
    for data in all_results.values():
        if data['plot_data']['relative']:
            max_depths.append(max(data['plot_data']['relative'].keys()))
    common_max_depth = max(max_depths) if max_depths else 1
    common_max_depth = min(common_max_depth, max_depth)
    
    # Define Rest color (uniform grey for all "Rest" entries)
    REST_COLOR = '#808080'  # Medium grey
    
    # Create a unified color mapping for all activities across all methods and depths
    # This ensures the same activity gets the same color in all plots
    all_activity_names = []
    for method_key, data in all_results.items():
        for depth in data['plot_data']['relative'].keys():
            df_depth = data['plot_data']['relative'][depth]
            for name in df_depth['activity_name']:
                if name not in all_activity_names and not str(name).startswith('Rest'):
                    all_activity_names.append(name)
    
    # Create color mapping
    main_colors = plt.cm.tab20(np.linspace(0, 1, 20)) # type: ignore
    color_map = {}
    color_idx = 0
    for name in all_activity_names:
        color_map[name] = main_colors[color_idx % len(main_colors)]
        color_idx += 1
    
    # 1. RELATIVE COMPARISON PLOTS (% contribution at each depth across methods)
    for depth in range(1, common_max_depth + 1):
        fig, ax = plt.subplots(figsize=(14, 8))
        
        method_names = []
        depth_data = []
        
        for method_key, data in all_results.items():
            if depth in data['plot_data']['relative']:
                df_depth = data['plot_data']['relative'][depth].copy()
                # Take top 8 contributors + rest
                df_depth_sorted = df_depth.sort_values('impact_pct', ascending=False)
                if len(df_depth_sorted) > 9:
                    rest_rows = df_depth_sorted[df_depth_sorted['activity_name'].str.startswith('Rest', na=False)]
                    top_rows = df_depth_sorted[~df_depth_sorted['activity_name'].str.startswith('Rest', na=False)].head(8)
                    df_depth_sorted = pd.concat([top_rows, rest_rows], ignore_index=True)
                
                # Shorten method name for display
                method_label = data['method_name'][:40]
                method_names.append(method_label)
                depth_data.append(df_depth_sorted)
        
        if not depth_data:
            continue
        
        # Create stacked bar chart with consistent colors
        bar_width = 0.7
        
        legend_handles = []
        legend_labels = []
        
        for i, (method_label, df_depth) in enumerate(zip(method_names, depth_data)):
            bottom = 0
            for j, (_, row) in enumerate(df_depth.iterrows()):
                label = row['activity_name'][:40] if row['activity_name'] else 'Unknown'
                value = max(0, row['impact_pct'])
                if value > 0.01:  # Only show if positive
                    # Check if this is a "Rest" entry
                    if 'Rest' in label or row['activity_name'].startswith('Rest'):
                        color = REST_COLOR
                    else:
                        # Use consistent color from the unified map
                        color = color_map.get(row['activity_name'], main_colors[j % len(main_colors)])
                    
                    bar = ax.bar(i, value, bar_width, bottom=bottom, 
                               color=color, edgecolor='black', linewidth=0.5)
                    bottom += value
                    
                    # Store for legend (only once per label)
                    if label not in legend_labels:
                        if 'Rest' in label and any('Rest' in lbl for lbl in legend_labels):
                            pass  # Skip duplicate Rest
                        else:
                            legend_handles.append(bar)
                            legend_labels.append(label)
                    
                    # Add value labels for significant contributions
                    if value > 8:
                        ax.text(i, bottom - value/2, f'{value:.1f}%', 
                               ha='center', va='center', fontsize=7, fontweight='bold')
        
        ax.set_xticks(range(len(method_names)))
        ax.set_xticklabels(method_names, rotation=45, ha='right', fontsize=9)
        ax.set_ylabel('Relative Contribution (%)', fontsize=12)
        ax.set_xlabel('Impact Methods', fontsize=12)
        ax.set_title(f'Relative Contributions at Depth {depth}\n(Cutoff: {min_pct}%)', fontweight='bold', fontsize=14)
        ax.set_ylim(0, 105)
        
        # Add legend with adjustable font size
        if legend_handles:
            ax.legend(legend_handles, legend_labels, loc='center left', 
                     bbox_to_anchor=(1, 0.5), fontsize=legend_fontsize, framealpha=0.9)
        
        ax.grid(True, alpha=0.3, axis='y')
        plt.tight_layout()
        plt.savefig(f'bw25_relative_comparison_depth_{depth}.svg', dpi=150, bbox_inches='tight')
        plt.show()
        print(f"\nPlot saved to bw25_relative_comparison_depth_{depth}.svg")
    
    # 2. ABSOLUTE COMPARISON PLOTS (MJ-eq at each depth across methods)
    for depth in range(1, common_max_depth + 1):
        fig, ax = plt.subplots(figsize=(14, 8))
        
        method_names = []
        depth_data_abs = []
        method_total_scores = []  # Store total scores for annotation
        
        for method_key, data in all_results.items():
            if depth in data['plot_data']['absolute']:
                df_depth = data['plot_data']['absolute'][depth].copy()
                df_depth_sorted = df_depth.sort_values('impact_abs', ascending=True) # False origin
                # Only filter out rows with near-zero impact to keep plot readable
                max_impact = df_depth_sorted['impact_abs'].max()
                df_depth_sorted = df_depth_sorted[
                    (df_depth_sorted['impact_abs'] > max_impact * 0.01) | 
                    (df_depth_sorted['activity_name'].str.startswith('Rest', na=False))
                ]
                
                method_label = data['method_name'][:40]
                method_names.append(method_label)
                depth_data_abs.append(df_depth_sorted)
                method_total_scores.append(data['total_score'])  # Store total score
        
        if not depth_data_abs:
            continue
        
        bar_width = 0.7
        legend_handles = []
        legend_labels = []
        
        # Collect all unique labels for consistent colors
        all_labels_abs = []
        for df_depth in depth_data_abs:
            for _, row in df_depth.iterrows():
                label = row['activity_name'][:40] if row['activity_name'] else 'Unknown'
                if label not in all_labels_abs and not label.startswith('Rest'):
                    all_labels_abs.append(label)
        
        # Create color mapping for absolute plots
        color_map_abs = {}
        color_idx = 0
        for name in all_labels_abs:
            color_map_abs[name] = main_colors[color_idx % len(main_colors)]
            color_idx += 1
        
        for i, (method_label, df_depth) in enumerate(zip(method_names, depth_data_abs)):
            bottom = 0
            # Store bar positions for percentage labels
            bar_positions = []
            bar_values = []
            bar_labels = []
            bar_colors_list = []
            
            for j, (_, row) in enumerate(df_depth.iterrows()):
                label = row['activity_name'][:40] if row['activity_name'] else 'Unknown'
                value = max(0, row['impact_abs'])
                if value > 1e-12:
                    # Check if this is a "Rest" entry
                    if 'Rest' in label or row['activity_name'].startswith('Rest'):
                        color = REST_COLOR
                    else:
                        color = color_map_abs.get(row['activity_name'], main_colors[j % len(main_colors)])
                    
                    bar = ax.bar(i, value, bar_width, bottom=bottom, 
                            color=color, edgecolor='black', linewidth=0.5)
                    bottom += value
                    
                    # Store for percentage labels
                    bar_positions.append(i)
                    bar_values.append(value)
                    bar_labels.append(label)
                    bar_colors_list.append(color)
                    
                    if label not in legend_labels:
                        if 'Rest' in label and any('Rest' in lbl for lbl in legend_labels):
                            pass  # Skip duplicate Rest
                        else:
                            legend_handles.append(bar)
                            legend_labels.append(label)
            
            # Add percentage labels on top of each bar
            total_for_method = method_total_scores[i]
            for idx, (pos, val, label) in enumerate(zip(bar_positions, bar_values, bar_labels)):
                if val > 1e-12:
                    pct = (val / total_for_method) * 100
                    # Only label if percentage is meaningful (>0.5%)
                    if pct > 0.5:
                        # Position label at the top of each segment
                        y_pos = 0
                        # Calculate cumulative position for this bar
                        for k in range(idx):
                            y_pos += bar_values[k]
                        # Add text label
                        ax.text(pos, y_pos + val/2, f'{pct:.1f}%', 
                            ha='center', va='center', fontsize=6, fontweight='bold', color='white')
        
        ax.set_xticks(range(len(method_names)))
        ax.set_xticklabels(method_names, rotation=45, ha='right', fontsize=9)
        ax.set_ylabel('Absolute Impact (MJ-eq)', fontsize=12)
        ax.set_xlabel('Impact Methods', fontsize=12)
        ax.set_title(f'Absolute Contributions at Depth {depth}\n(Cutoff: {min_pct}%)', fontweight='bold', fontsize=14)
        ax.set_yscale('log')
        
        # Add total score annotations below x-axis labels
        for i, (method_label, total_score) in enumerate(zip(method_names, method_total_scores)):
            ax.text(i, -0.15, f'Total: {total_score:.2e}', 
                    ha='center', va='top', fontsize=8, transform=ax.get_xaxis_transform(), 
                    rotation=0, color='black', fontweight='bold')
        
        # Add legend with adjustable font size
        if legend_handles:
            ax.legend(legend_handles, legend_labels, loc='center left', 
                    bbox_to_anchor=(1, 0.5), fontsize=legend_fontsize, framealpha=0.9)
        
        ax.grid(True, alpha=0.3, axis='y')
        plt.tight_layout()
        plt.savefig(f'bw25_absolute_comparison_depth_{depth}.svg', dpi=150, bbox_inches='tight')
        plt.show()
        print(f"\nPlot saved to bw25_absolute_comparison_depth_{depth}.svg")

    # 2.5 ABSOLUTE MJ-EQ COMPARISON PLOTS (Linear Scale)
    for depth in range(1, common_max_depth + 1):
        fig, ax = plt.subplots(figsize=(14, 8))

        method_names = []
        depth_data_abs = []
        method_total_scores = []

        for method_key, data in all_results.items():
            if depth in data['plot_data']['absolute']:
                df_depth = data['plot_data']['absolute'][depth].copy()
                df_depth_sorted = df_depth.sort_values('impact_abs', ascending=False)
                max_impact = df_depth_sorted['impact_abs'].max()
                df_depth_sorted = df_depth_sorted[
                    (df_depth_sorted['impact_abs'] > max_impact * 0.01) |
                    (df_depth_sorted['activity_name'].str.startswith('Rest', na=False))
                ]
                method_label = data['method_name'][:40]
                method_names.append(method_label)
                depth_data_abs.append(df_depth_sorted)
                method_total_scores.append(data['total_score'])

        if not depth_data_abs:
            continue

        bar_width = 0.7
        legend_handles = []
        legend_labels = []

        all_labels_abs = []
        for df_depth in depth_data_abs:
            for _, row in df_depth.iterrows():
                label = row['activity_name'][:40]
                if label not in all_labels_abs and not str(label).startswith('Rest'):
                    all_labels_abs.append(label)

        color_map_abs = {}
        for idx, name in enumerate(all_labels_abs):
            color_map_abs[name] = main_colors[idx % len(main_colors)]

        for i, (method_label, df_depth) in enumerate(zip(method_names, depth_data_abs)):
            bottom = 0
            bar_positions = []
            bar_values = []
            bar_labels = []

            for j, (_, row) in enumerate(df_depth.iterrows()):
                label = row['activity_name'][:40]
                value = max(0, row['impact_abs'])
                if value > 1e-12:
                    color = REST_COLOR if ('Rest' in label or row['activity_name'].startswith('Rest')) else color_map_abs.get(row['activity_name'], main_colors[j % len(main_colors)])
                    bar = ax.bar(i, value, bar_width, bottom=bottom, color=color, edgecolor='black', linewidth=0.5)
                    bottom += value
                    bar_positions.append(i)
                    bar_values.append(value)
                    bar_labels.append(label)
                    if label not in legend_labels:
                        if not ('Rest' in label and any('Rest' in lbl for lbl in legend_labels)):
                            legend_handles.append(bar)
                            legend_labels.append(label)

            total_for_method = method_total_scores[i]
            for idx, (pos, val, label) in enumerate(zip(bar_positions, bar_values, bar_labels)):
                if val > 1e-12:
                    pct = (val / total_for_method) * 100
                    if pct > 10:
                        y_pos = sum(bar_values[:idx])
                        ax.text(pos, y_pos + val/2, f'{pct:.1f}%', ha='center', va='center', fontsize=6, fontweight='bold', color='white')

        ax.set_xticks(range(len(method_names)))
        ax.set_xticklabels(method_names, rotation=45, ha='right', fontsize=9)
        ax.set_ylabel('Absolute Impact (MJ-eq)', fontsize=12)
        ax.set_xlabel('Impact Methods', fontsize=12)
        ax.set_title(f'Absolute Contributions at Depth {depth} (Linear Scale)\n(Cutoff: {min_pct}%)', fontweight='bold', fontsize=14)

        for i, total_score in enumerate(method_total_scores):
            ax.text(i, -0.15, f'Total: {total_score:.2e}', ha='center', va='top', fontsize=8, transform=ax.get_xaxis_transform(), rotation=0, color='black', fontweight='bold')

        if legend_handles:
            ax.legend(legend_handles, legend_labels, loc='center left', bbox_to_anchor=(1, 0.5), fontsize=legend_fontsize, framealpha=0.9)

        ax.grid(True, alpha=0.3, axis='y')
        plt.tight_layout()
        plt.savefig(f'bw25_absolute_linear_depth_{depth}.svg', dpi=150, bbox_inches='tight')
        plt.show()
        print(f"\nPlot saved to bw25_absolute_linear_depth_{depth}.svg")
    
    # 3. HEATMAP showing total scores across methods
    if len(all_total_scores) > 1:
        fig, ax = plt.subplots(figsize=(10, max(4, len(all_total_scores) * 0.5)))
        
        method_labels = [data['method_name'][:50] for data in all_results.values()]
        scores = list(all_total_scores.values())
        
        # Handle zero or negative scores for log transform
        scores_for_log = [max(s, 1e-12) for s in scores]
        scores_norm = np.log10(scores_for_log)
        
        im = ax.imshow([scores_norm], aspect='auto', cmap='YlOrRd')
        ax.set_yticks([])
        ax.set_xticks(range(len(method_labels)))
        ax.set_xticklabels(method_labels, rotation=45, ha='right', fontsize=9)
        ax.set_title('Total Impact Scores Across Methods (log10(MJ-eq))', fontweight='bold', fontsize=12)
        
        for i, (score, score_norm) in enumerate(zip(scores, scores_norm)):
            color = 'white' if score_norm > np.median(scores_norm) else 'black'
            ax.text(i, 0, f'{score:.2e}', ha='center', va='center', color=color, fontsize=9)
        
        cbar = plt.colorbar(im, label='log10(Impact in MJ-eq)')
        cbar.ax.tick_params(labelsize=legend_fontsize)
        
        plt.tight_layout()
        plt.savefig('bw25_total_scores_heatmap.svg', dpi=150, bbox_inches='tight')
        plt.show()
        print("\nPlot saved to bw25_total_scores_heatmap.svg")
        
    
    return all_results, all_total_scores

### usage

In [ ]:
# ============================================================================
# Contribution Tree Analysis for Copper Cathode Production Activities
# ============================================================================
import bw2data as bd
import matplotlib.pyplot as plt
import matplotlib
from bw2calc import LCA
# Assuming contribution_tree_bw25 is imported from your module

# Set matplotlib parameters
matplotlib.rcParams['font.size'] = 14

# Access your database
db = bd.Database(technosphere_db_name)  # Use your technosphere database name

# ----------------------------------------------------------------------------
# Filter activities: reference product = "copper, cathode", exclude markets and treatment
# ----------------------------------------------------------------------------
all_activities = list(db) # type: ignore

copper_cathode_activities = []
for act in all_activities:
    ref_prod = act.get('reference product', '')
    
    if ref_prod == 'copper, cathode':
        upr_name = act.get('name', '')
        # Exclude activities that have "market" or "treatment" in the unit process name
        if 'market' not in upr_name.lower() and 'treatment' not in upr_name.lower():
            copper_cathode_activities.append(act)

if not copper_cathode_activities:
    raise ValueError("No 'copper, cathode' production activities found (excluding markets and treatment).")

print(f"Found {len(copper_cathode_activities)} activities with reference product 'copper, cathode' (markets and treatment excluded).")

# Define methods to evaluate
methods_to_evaluate = [
    ('Cumulative Exergy Demand (CExD)', 'non-renewable: total', 'exergy content')
]

# ----------------------------------------------------------------------------
# Run contribution tree analysis for each filtered activity
# ----------------------------------------------------------------------------
for act in copper_cathode_activities:
    print(f"\n{'='*60}")
    print(f"Activity: {act['name']}")
    print(f"Reference product: {act.get('reference product')}")
    print(f"Location: {act.get('location')}")
    print(f"Key: {act.key}")
    print(f"{'='*60}\n")
    
    # Run the contribution tree analysis
    try:
        all_results, all_scores = contribution_tree_bw25(
            fu_activity=act,
            methods=methods_to_evaluate,
            max_depth=3,
            min_pct=2.50,
            legend_fontsize=10  # Adjust this value to change legend text size
        )
        
        # Access individual results
        for method_key, data in all_results.items():
            method_name = " | ".join(str(x) for x in method_key[-2:])
            print(f"\n{method_name}:")
            print(f"  Total score: {data['total_score']:.6e} MJ-eq")
            print(f"  Number of contributions: {len(data['df'])}")
            print(f"  Top contributions:")
            # Display top 5 contributions
            top_contrib = data['df'].head(5)
            for idx, row in top_contrib.iterrows():
                print(f"    {row['name']}: {row['cumulative contribution']:.2%}")
            
    except Exception as e:
        print(f"Error running contribution tree for {act['name']}: {e}")

print("\n" + "="*60)
print("Contribution tree analysis completed for all activities.")

Found 11 activities with reference product 'copper, cathode' (markets and treatment excluded).

Activity: platinum group metal mine operation, ore with high palladium content
Reference product: copper, cathode
Location: RU
Key: ('ecoinvent-3.12-cutoff', '7383ffa092989743b144e22c0f11efab')


########################################################################################################################
# Analyzing method: non-renewable: total | exergy content
# Method key: ('Cumulative Exergy Demand (CExD)', 'non-renewable: total', 'exergy content')
########################################################################################################################



# 🔚 end